# 03 Inference

This notebook performs:
1. one confidence interval and one one-sample test for a population proportion
2. one confidence interval and one one-sample t-test for a population mean


In [1]:
import math
import pandas as pd
from scipy import stats

df = pd.read_csv('../data/raw/YRBS_2007.csv')

def recode_smoking(x):
    if pd.isna(x):
        return pd.NA
    elif x in [2, 3, 4, 5, 6, 7]:
        return 1
    elif x == 1:
        return 0
    else:
        return pd.NA

smoke_raw = df['CurrentCigaretteUse']
df['CurrentCigaretteUse_Bin'] = df['CurrentCigaretteUse'].apply(recode_smoking)
smoke_bin = df['CurrentCigaretteUse_Bin'].dropna()
height = df['HowTallAreYouWithoutShoesInMeters'].dropna()

## Research Questions
1. Is the proportion of students who currently smoke different from 0.20?
2. Is the mean height of students different from 1.70 meters?


In [2]:
# Proportion inference
p0 = 0.20
n_p = len(smoke_bin)
x = int(smoke_bin.sum())
phat = x / n_p

z_critical = stats.norm.ppf(0.975)
se_phat = math.sqrt(phat * (1 - phat) / n_p)
ci_prop = (
    phat - z_critical * se_phat,
    phat + z_critical * se_phat
)

se0 = math.sqrt(p0 * (1 - p0) / n_p)
z_stat = (phat - p0) / se0
p_value_prop = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print('=== Proportion Analysis ===')
print(f'n = {n_p}')
print(f'x = {x}')
print(f'p-hat = {phat:.6f}')
print(f'95% CI = ({ci_prop[0]:.6f}, {ci_prop[1]:.6f})')
print(f'H0: p = {p0}')
print(f'Ha: p != {p0}')
print(f'z = {z_stat:.6f}')
print(f'p-value = {p_value_prop:.6f}')
print('Decision:', 'Reject H0' if p_value_prop < 0.05 else 'Fail to reject H0')

=== Proportion Analysis ===
n = 13323
x = 2589
p-hat = 0.194326
95% CI = (0.187607, 0.201044)
H0: p = 0.2
Ha: p != 0.2
z = -1.637423
p-value = 0.101542
Decision: Fail to reject H0


### Interpretation
- This analysis estimates the population proportion of students who currently smoke.
- The confidence interval gives a plausible range for the true smoking proportion.
- The hypothesis test checks whether the proportion is statistically different from 0.20.


In [3]:
# Mean inference
mu0 = 1.70
n_m = len(height)
xbar = height.mean()
s = height.std(ddof=1)
se_mean = s / math.sqrt(n_m)

t_critical = stats.t.ppf(0.975, df=n_m - 1)
ci_mean = (
    xbar - t_critical * se_mean,
    xbar + t_critical * se_mean
)

t_stat = (xbar - mu0) / se_mean
p_value_mean = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_m - 1))

print('=== Mean Analysis ===')
print(f'n = {n_m}')
print(f'x-bar = {xbar:.6f}')
print(f's = {s:.6f}')
print(f'95% CI = ({ci_mean[0]:.6f}, {ci_mean[1]:.6f})')
print(f'H0: mu = {mu0}')
print(f'Ha: mu != {mu0}')
print(f't = {t_stat:.6f}')
print(f'p-value = {p_value_mean:.12f}')
print('Decision:', 'Reject H0' if p_value_mean < 0.05 else 'Fail to reject H0')

=== Mean Analysis ===
n = 13062
x-bar = 1.694038
s = 0.101466
95% CI = (1.692297, 1.695778)
H0: mu = 1.7
Ha: mu != 1.7
t = -6.715846
p-value = 0.000000000019
Decision: Reject H0


### Interpretation
- This analysis estimates the population mean height.
- The confidence interval gives a plausible range for the true mean height.
- The one-sample t-test checks whether the mean height is statistically different from 1.70 meters.


In [4]:
summary = pd.DataFrame({
    'Analysis': ['Population Proportion', 'Population Mean'],
    'Variable': ['CurrentCigaretteUse', 'HowTallAreYouWithoutShoesInMeters'],
    'Benchmark': [p0, mu0],
    'Sample Size': [n_p, n_m],
    'Estimate': [phat, xbar],
    '95% CI Lower': [ci_prop[0], ci_mean[0]],
    '95% CI Upper': [ci_prop[1], ci_mean[1]],
    'Test Statistic': [z_stat, t_stat],
    'p-value': [p_value_prop, p_value_mean],
    'Decision (alpha=0.05)': [
        'Reject H0' if p_value_prop < 0.05 else 'Fail to reject H0',
        'Reject H0' if p_value_mean < 0.05 else 'Fail to reject H0'
    ]
})

display(summary)
summary.to_csv('../outputs/tables/inference_summary_table.csv', index=False)

,Analysis,Variable,Benchmark,Sample Size,Estimate,95% CI Lower,95% CI Upper,Test Statistic,p-value,Decision (alpha=0.05)
0,Population Proportion,CurrentCigaretteUse,0.2,13323,0.194326,0.187607,0.201044,-1.637423,1.015422e-01,Fail to reject H0
1,Population Mean,HowTallAreYouWithoutShoesInMeters,1.7,13062,1.694038,1.692297,1.695778,-6.715846,1.947176e-11,Reject H0


## Final Synthesis
- Research question: whether the smoking proportion differs from 0.20, and whether the mean height differs from 1.70 m.
- EDA findings: the smoking variable is imbalanced, and the height variable is roughly bell-shaped with some outliers.
- Inferential results: report the confidence intervals, p-values, and final decisions here after running the notebook.
- Final conclusion: write a short context-based summary that connects the EDA and inference results.
